In [12]:
import matplotlib.pyplot as plt
import nelpy as nel
import nelpy.plotting as npl
import numpy as np
import seaborn as sns
from neuro_py.io import loading
from neuro_py.plotting.figure_helpers import set_size
from neuro_py.tuning import maps
from skimage import measure
import os
import pandas as pd
import scipy.io as sio
import logging

logging.getLogger().setLevel(logging.ERROR)
basepath =  r"S:\data\latentseq\RM10\rm10_day11_20250815"
basename=os.path.basename(basepath)# Define the new folder path
#create working folder
new_folder_path = os.path.join(basepath, "complex_system")
brain_region="CA1"
animal="Rat"   #or Mouse
cell_type='all'
path_parts = basepath.split("\\")  # Split the path using backslash
animal_name = path_parts[-2]  # Second-to-last part is the animal name
day = path_parts[-1]  # Last part is the day
save_dir = r"D:\Documents\GitHub\R_Project\PPP\codebook\Criticality_analysis\complex_system\manifold"
if not os.path.exists(save_dir):
    os.makedirs(save_dir)
# Create the folder if it doesn't exist
if not os.path.exists(new_folder_path):
    os.makedirs(new_folder_path)
    print(f"Folder created: {new_folder_path}")
else:
    print(f"Folder already exists: {new_folder_path}")
# load in spike data from hpc pyramidal cells
st, cm = loading.load_spikes(
    basepath, brainRegion=brain_region                          # putativeCellType="Pyr"
)
# load theta epochs
state_dict = loading.load_SleepState_states(basepath)
nrem_epochs = nel.EpochArray(
    state_dict["NREMstate"],
)
awake_epochs = nel.EpochArray(
    state_dict["WAKEstate"],
)
# theta_epochs = nel.EpochArray(
#     state_dict["THETA"],
# )
rem_epochs=nel.EpochArray(state_dict["REMstate"],)
try:
    # Try default loader
    epoch_df = loading.load_epoch(basepath)
except Exception as e:
    print(f"load_epoch failed with error: {e}")
    print("Falling back to manual .mat file load...")

    # Load the .mat file
    mat = sio.loadmat(os.path.join(basepath, "DayNight25.MergePoints.events.mat"), squeeze_me=True)

    # Extract the 'MergePoints' structure
    merge_points = mat['MergePoints']

    # Extract the timestamps array: shape (N, 2)
    timestamps = merge_points['timestamps']
    # Unwrap the array inside
    timestamps = np.array(timestamps.tolist())

    epoch_df = pd.DataFrame(timestamps, columns=["startTime", "stopTime"])

sleep_epochs = nel.EpochArray(np.array([epoch_df.startTime, epoch_df.stopTime]).T)
E_spikes = st.time[(cm.putativeCellType == 'Pyramidal Cell')]
I_spikes= st.time[(cm.putativeCellType.str.contains("Interneuron"))]
spike_times=st.time
n_cell=len(spike_times)
print(f"number of cells: {n_cell}")

Folder already exists: W:\data\PPP\PPP20\day5\complex_system
number of cells: 297


In [13]:
import os, re, numpy as np, pandas as pd
import glob
def _load_mat_any(mat_path):
    from scipy.io import loadmat
    try:
        return loadmat(
            mat_path,
            squeeze_me=True,
            struct_as_record=False,
            simplify_cells=True
        )
    except TypeError:
        return loadmat(
            mat_path,
            squeeze_me=True,
            struct_as_record=False
        )
    except NotImplementedError:
        import mat73
        return mat73.loadmat(mat_path)
def _to_1d_array(x):
    """
    Robustly coerce MATLAB-loaded field to a 1D numpy array.
    - Scalars -> 1-length array
    - Lists/tuples/ndarrays -> flattened 1D
    - None / missing -> empty array
    """
    if x is None:
        return np.array([], dtype=float)
    arr = np.asarray(x, dtype=object)  # start as object to avoid surprises
    if arr.ndim == 0:
        # scalar or unsized object
        return np.atleast_1d(arr).astype(float, copy=False)
    # flatten and try float cast (ignore non-numeric gracefully)
    try:
        return np.asarray(arr).ravel().astype(float, copy=False)
    except (ValueError, TypeError):
        # fallback: best-effort, convert elementwise
        return np.array([float(v) for v in np.ravel(arr)], dtype=float)

def _to_state_str(s):
    # MATLAB char arrays sometimes come in as numpy arrays
    try:
        if isinstance(s, (bytes, bytearray)):
            return s.decode('utf-8', 'ignore')
        if isinstance(s, str):
            return s
        # arrays or weird containers -> join characters
        arr = np.asarray(s)
        if arr.ndim == 0:
            return str(arr.item())
        # e.g., array of characters
        return ''.join(arr.tolist()) if arr.dtype.kind in ('U','S') else str(arr)
    except Exception:
        return str(s)

def load_d2_timeline_complete(basepath, region):
    """Complete version that loads ALL sleep-state data, including d2_err."""
    region = region.upper()
    search_dir = os.path.join(basepath, "complex_system")
    # pats = [f"*_{region}_D2_time_resolved_rippleablate.mat", f"*_{region}_D2_time_rippleablate.mat"]
    pats = [f"*_{region}_D2_time_resolvedforplot.mat", f"*_{region}_D2_timeforplot.mat"]
    files = []
    for pat in pats:
        files.extend(glob.glob(os.path.join(search_dir, pat)))
    if not files:
        raise FileNotFoundError(f"No *_{region}_D2_time*.mat under {search_dir}")

    mat_path = max(files, key=os.path.getmtime)
    print(f"Loading: {os.path.basename(mat_path)}")

    mat = _load_mat_any(mat_path)
    results = mat['results']
    params = mat.get('params', {})

    # Make results iterable no matter if it's a list, ndarray, or dict
    if isinstance(results, dict):
        iter_results = list(results.values())
    else:
        iter_results = list(results if isinstance(results, (list, tuple)) else np.atleast_1d(results))

    print(f"📊 Found {len(iter_results)} result items (sleep states)")
    print(f"📊 Parameters: {params}")

    # Try to infer animal/day from filename if available
    fname = os.path.basename(mat_path)
    animal_name = None
    day = None
    m_day = re.search(r'day(\d+)', fname, flags=re.I)
    if m_day:
        day = int(m_day.group(1))
    m_animal = re.match(r'([^_]+)_day\d+_', fname)
    if m_animal:
        animal_name = m_animal.group(1)

    all_centers, all_d2, all_d2err, all_states = [], [], [], []
    n_states_with_err = 0

    for i, result_item in enumerate(iter_results):
        # Access as dict-like regardless of MATLAB struct quirks
        state = _to_state_str(result_item.get('state', 'UNKNOWN')) if hasattr(result_item, 'get') else \
                _to_state_str(result_item['state']) if (isinstance(result_item, dict) or hasattr(result_item, '__getitem__')) else 'UNKNOWN'

        centers_raw = result_item.get('center', []) if hasattr(result_item, 'get') else \
                      (result_item['center'] if (isinstance(result_item, dict) or hasattr(result_item, '__getitem__')) and 'center' in result_item else [])
        d2_raw = result_item.get('d2', []) if hasattr(result_item, 'get') else \
                 (result_item['d2'] if (isinstance(result_item, dict) or hasattr(result_item, '__getitem__')) and 'd2' in result_item else [])
        d2err_raw = result_item.get('d2_err', []) if hasattr(result_item, 'get') else \
                    (result_item['d2_err'] if (isinstance(result_item, dict) or hasattr(result_item, '__getitem__')) and 'd2_err' in result_item else [])

        centers  = _to_1d_array(centers_raw)
        d2_vals  = _to_1d_array(d2_raw)
        d2_err   = _to_1d_array(d2err_raw)

        # Presence flag (some states may not have d2_err)
        has_err = d2_err.size > 0 and np.any(np.isfinite(d2_err))
        n_states_with_err += int(has_err)

        # Harmonize lengths: truncate all to the common min
        sizes = [centers.size, d2_vals.size] + ([d2_err.size] if has_err else [])
        n_pts = min(sizes) if min(sizes) > 0 else 0
        print(f"   State {i}: {state} - requested points={centers.size}, actual used={n_pts}")

        if n_pts == 0:
            print("     ⚠️ Empty state - no data points")
            continue

        centers = centers[:n_pts]
        d2_vals = d2_vals[:n_pts]
        d2_err  = (d2_err[:n_pts] if has_err else np.full(n_pts, np.nan, dtype=float))

        # Ranges
        try:
            print(f"     Time range: {float(np.nanmin(centers))} to {float(np.nanmax(centers))}")
        except ValueError:
            print("     Time range: (all NaN)")

        try:
            print(f"     D2 range: {float(np.nanmin(d2_vals)):.4f} to {float(np.nanmax(d2_vals)):.4f}")
        except ValueError:
            print("     D2 range: (all NaN)")

        if np.any(np.isfinite(d2_err)):
            try:
                dmin, dmax = float(np.nanmin(d2_err)), float(np.nanmax(d2_err))
                print(f"     D2_ERR range: {dmin:.4f} to {dmax:.4f}")
            except ValueError:
                print("     D2_ERR range: (all NaN)")
        else:
            print("     D2_ERR: missing → filled with NaN")

        all_centers.append(centers)
        all_d2.append(d2_vals)
        all_d2err.append(d2_err)
        all_states.append(np.array([state]*n_pts, dtype=object))

    if len(all_centers) == 0:
        raise ValueError("No data found in any sleep state - all states are empty")

    # Concatenate
    all_centers = np.concatenate(all_centers)
    all_d2      = np.concatenate(all_d2)
    all_d2err   = np.concatenate(all_d2err)
    all_states  = np.concatenate(all_states)

    # DataFrame (include d2_err)
    df = pd.DataFrame({
        'center_s': all_centers.astype(float, copy=False),
        'd2':       all_d2.astype(float, copy=False),
        'd2_err':   all_d2err.astype(float, copy=False),
        'state':    all_states.astype(str)
    }).sort_values('center_s').reset_index(drop=True)

    print(f"\n✅ Complete timeline created:")
    print(f"📊 Total data points: {len(df)}")
    print(f"📊 Time range: {df['center_s'].min()} to {df['center_s'].max()}")
    print(f"📊 Duration: {(df['center_s'].max() - df['center_s'].min()):.1f} seconds")
    print(f"📊 State distribution:")
    print(df['state'].value_counts())

    # Empty-state tracker
    empty_states = []
    for ri in iter_results:
        st = _to_state_str(ri.get('state','UNKNOWN')) if hasattr(ri,'get') else \
             _to_state_str(ri['state']) if (isinstance(ri,dict) or hasattr(ri,'__getitem__')) and 'state' in ri else 'UNKNOWN'
        csz = np.size(np.atleast_1d(ri.get('center', []))) if hasattr(ri,'get') else \
              np.size(np.atleast_1d(ri['center'])) if (isinstance(ri,dict) or hasattr(ri,'__getitem__')) and 'center' in ri else 0
        if csz == 0:
            empty_states.append(st)

    meta = {
        "animal_name": animal_name,
        "day": day,
        "region": region,
        "path": mat_path,
        "params": params,
        "columns": list(df.columns),
        "shape": df.shape,
        "n_states": len(iter_results),
        "states_with_d2_err": n_states_with_err,
        "empty_states": empty_states
    }

    return df, meta



# Load the CA1 d2 timeline
try:
    ca1_complete_df, ca1_complete_meta = load_d2_timeline_complete(basepath, "CA1")
    
    # pfc_complete_df, pfc_complete_meta = load_d2_timeline_complete(basepath, "PFC")

    
except Exception as e:
    print(f"❌ Failed: {e}")
    import traceback
    traceback.print_exc()

print("\n" + "="*60)

❌ Failed: No *_CA1_D2_time*.mat under W:\data\PPP\PPP20\day5\complex_system



Traceback (most recent call last):
  File "C:\Users\Cornell\AppData\Local\Temp\ipykernel_70220\39668419.py", line 210, in <module>
    ca1_complete_df, ca1_complete_meta = load_d2_timeline_complete(basepath, "CA1")
                                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Cornell\AppData\Local\Temp\ipykernel_70220\39668419.py", line 67, in load_d2_timeline_complete
    raise FileNotFoundError(f"No *_{region}_D2_time*.mat under {search_dir}")
FileNotFoundError: No *_CA1_D2_time*.mat under W:\data\PPP\PPP20\day5\complex_system


In [14]:
import numpy as np
from scipy.signal import welch
import nelpy as nel

def _flatten_spike_times(spike_times):
    """Accepts list/tuple/array of arrays or a single 1D array, returns sorted 1D float array."""
    if isinstance(spike_times, (list, tuple)):
        # list of arrays or scalars
        if len(spike_times) == 0:
            return np.array([], dtype=float)
        if hasattr(spike_times[0], "__len__") and not np.isscalar(spike_times[0]):
            times = np.concatenate([np.asarray(st, dtype=float).ravel()
                                    for st in spike_times if st is not None and len(st) > 0])
        else:
            times = np.asarray(spike_times, dtype=float).ravel()
    else:
        arr = np.asarray(spike_times, dtype=object)
        if arr.dtype == object:
            times = np.concatenate([np.asarray(st, dtype=float).ravel()
                                    for st in arr if st is not None and len(st) > 0])
        else:
            times = arr.astype(float).ravel()

    times = times[np.isfinite(times)]
    if times.size == 0:
        raise ValueError("No valid spike times found after flattening.")
    return np.sort(times)

def bin_spikes(spike_times, bin_size=0.04, t_start=None, t_stop=None):
    st = _flatten_spike_times(spike_times)
    if t_start is None: t_start = float(np.floor(st.min()))
    if t_stop  is None: t_stop  = float(np.ceil(st.max()))
    edges = np.arange(t_start, t_stop + bin_size, bin_size)
    counts, _ = np.histogram(st, bins=edges)
    rates = counts / bin_size
    t = edges[:-1] + bin_size/2
    return t, rates

# def segments_from_epochs(t, x, epochs):
#     arr = epochs.data if isinstance(epochs, nel.EpochArray) else np.asarray(epochs)
#     return [x[(t >= s) & (t < e)] for s, e in arr if np.any((t >= s) & (t < e))]

In [15]:
# Create plot for random 200 second segment containing all states
import matplotlib.pyplot as plt
import numpy as np
import random

# First compute MUA from spike times
print("Computing MUA firing rate from spike times...")
print("="*50)

# Compute MUA firing rate with 0.04 second bins
time_bins, firing_rate = bin_spikes(spike_times, bin_size=0.04)

Computing MUA firing rate from spike times...


In [5]:
import numpy as np
import pandas as pd

# --- Inputs assumed to exist ---
# spike_times
# time_bins, firing_rate  # from bin_spikes(..., bin_size=0.04)
# ca1_complete_df          # has a 'center_s' column

# 1) Make sure we have bin *centers* that match firing_rate length
tb = np.asarray(time_bins, float).ravel()
fr = np.asarray(firing_rate, float).ravel()

if tb.size == fr.size + 1:            # looks like edges → make centers
    t = 0.5 * (tb[:-1] + tb[1:])
elif tb.size == fr.size:               # already centers
    t = tb
else:
    raise ValueError("time_bins length must equal firing_rate length or be exactly one longer (edges).")

# 2) Clean + sort
ok = np.isfinite(t) & np.isfinite(fr)
t, fr = t[ok], fr[ok]
ordr = np.argsort(t)
t, fr = t[ordr], fr[ordr]

# 3) 30 s centered moving average of MUA (Hz)
dt = float(np.median(np.diff(t)))
win_s = 30.0
win_n = max(int(round(win_s / dt)), 1)

# Use pandas rolling for a centered window; allow some tolerance near edges
fr_series = pd.Series(fr, index=pd.Index(t, name="t"))
fr_30s = fr_series.rolling(window=win_n, center=True, min_periods=max(1, win_n//4)).mean().to_numpy()

# 4) Interpolate the smoothed MUA onto the D² centers
centers = np.asarray(ca1_complete_df["center_s"], float)
mua_at_centers = np.interp(centers, t, fr_30s, left=np.nan, right=np.nan)

# 5) Save to dataframe (Hz)
ca1_complete_df["MUA"] = mua_at_centers


NameError: name 'ca1_complete_df' is not defined

SWR and barrage

In [17]:
import numpy as np
import scipy.io as sio      
import scipy
# Load ripple events
ripple_events = loading.load_ripples_events(basepath=basepath)
ripple_epochs = nel.EpochArray(np.array([ripple_events.start, ripple_events.stop]).T)
ripple_peaks=ripple_events.peaks
    # Load barrage events
ripple_epochs=nrem_epochs&ripple_epochs
try:
    # Load barrage events
    #brpath=r"W:\data\PPP\CA2\PPP7\day8"
    barrage_events = loading.load_barrage_events(basepath=basepath)  
    # Create barrage epochs
    barrage_epochs = nel.EpochArray(np.array([barrage_events.start, barrage_events.stop]).T)
    # Find barrages in NREM epochs
    barrages_in_nrem = barrage_epochs & nrem_epochs
    mat_path = os.path.join(basepath, f'{basename}.HSEn2.events.mat')

    data = scipy.io.loadmat(mat_path, struct_as_record=False, squeeze_me=True)
    barrage = data['HSEn2']
    barrage_timestamps= barrage.timestamps 
    barrage_epochs=nrem_epochs&barrage_epochs
except Exception as e:
    # General exception for unexpected errors
    print(f"No barrage file: {e}")
    barrage_epochs=[]
    # Extract start times

No barrage file: 'DataFrame' object has no attribute 'start'


D:\Documents\GitHub\neuro_py\neuro_py\io\loading.py:1036: UserWarning: No barrage file found for W:\data\PPP\PPP20\day5


load d2 and error

In [41]:
# import os, re, numpy as np, pandas as pd
# import glob
# def _load_mat_any(mat_path):
#     from scipy.io import loadmat
#     try:
#         return loadmat(
#             mat_path,
#             squeeze_me=True,
#             struct_as_record=False,
#             simplify_cells=True
#         )
#     except TypeError:
#         return loadmat(
#             mat_path,
#             squeeze_me=True,
#             struct_as_record=False
#         )
#     except NotImplementedError:
#         import mat73
#         return mat73.loadmat(mat_path)
# def _to_1d_array(x):
#     """
#     Robustly coerce MATLAB-loaded field to a 1D numpy array.
#     - Scalars -> 1-length array
#     - Lists/tuples/ndarrays -> flattened 1D
#     - None / missing -> empty array
#     """
#     if x is None:
#         return np.array([], dtype=float)
#     arr = np.asarray(x, dtype=object)  # start as object to avoid surprises
#     if arr.ndim == 0:
#         # scalar or unsized object
#         return np.atleast_1d(arr).astype(float, copy=False)
#     # flatten and try float cast (ignore non-numeric gracefully)
#     try:
#         return np.asarray(arr).ravel().astype(float, copy=False)
#     except (ValueError, TypeError):
#         # fallback: best-effort, convert elementwise
#         return np.array([float(v) for v in np.ravel(arr)], dtype=float)

# def _to_state_str(s):
#     # MATLAB char arrays sometimes come in as numpy arrays
#     try:
#         if isinstance(s, (bytes, bytearray)):
#             return s.decode('utf-8', 'ignore')
#         if isinstance(s, str):
#             return s
#         # arrays or weird containers -> join characters
#         arr = np.asarray(s)
#         if arr.ndim == 0:
#             return str(arr.item())
#         # e.g., array of characters
#         return ''.join(arr.tolist()) if arr.dtype.kind in ('U','S') else str(arr)
#     except Exception:
#         return str(s)

# def load_d2_timeline_complete(basepath, region):
#     """Complete version that loads ALL sleep-state data, including d2_err."""
#     region = region.upper()
#     search_dir = os.path.join(basepath, "complex_system")
#     pats = [f"*_{region}_D2_time_resolved_error_bar.mat", f"*_{region}_D2_time_error_bar.mat"]

#     files = []
#     for pat in pats:
#         files.extend(glob.glob(os.path.join(search_dir, pat)))
#     if not files:
#         raise FileNotFoundError(f"No *_{region}_D2_time*.mat under {search_dir}")

#     mat_path = max(files, key=os.path.getmtime)
#     print(f"Loading: {os.path.basename(mat_path)}")

#     mat = _load_mat_any(mat_path)
#     results = mat['results']
#     params = mat.get('params', {})

#     # Make results iterable no matter if it's a list, ndarray, or dict
#     if isinstance(results, dict):
#         iter_results = list(results.values())
#     else:
#         iter_results = list(results if isinstance(results, (list, tuple)) else np.atleast_1d(results))

#     print(f"📊 Found {len(iter_results)} result items (sleep states)")
#     print(f"📊 Parameters: {params}")

#     # Try to infer animal/day from filename if available
#     fname = os.path.basename(mat_path)
#     animal_name = None
#     day = None
#     m_day = re.search(r'day(\d+)', fname, flags=re.I)
#     if m_day:
#         day = int(m_day.group(1))
#     m_animal = re.match(r'([^_]+)_day\d+_', fname)
#     if m_animal:
#         animal_name = m_animal.group(1)

#     all_centers, all_d2, all_d2err, all_states = [], [], [], []
#     n_states_with_err = 0

#     for i, result_item in enumerate(iter_results):
#         # Access as dict-like regardless of MATLAB struct quirks
#         state = _to_state_str(result_item.get('state', 'UNKNOWN')) if hasattr(result_item, 'get') else \
#                 _to_state_str(result_item['state']) if (isinstance(result_item, dict) or hasattr(result_item, '__getitem__')) else 'UNKNOWN'

#         centers_raw = result_item.get('center', []) if hasattr(result_item, 'get') else \
#                       (result_item['center'] if (isinstance(result_item, dict) or hasattr(result_item, '__getitem__')) and 'center' in result_item else [])
#         d2_raw = result_item.get('d2', []) if hasattr(result_item, 'get') else \
#                  (result_item['d2'] if (isinstance(result_item, dict) or hasattr(result_item, '__getitem__')) and 'd2' in result_item else [])
#         d2err_raw = result_item.get('d2_err', []) if hasattr(result_item, 'get') else \
#                     (result_item['d2_err'] if (isinstance(result_item, dict) or hasattr(result_item, '__getitem__')) and 'd2_err' in result_item else [])

#         centers  = _to_1d_array(centers_raw)
#         d2_vals  = _to_1d_array(d2_raw)
#         d2_err   = _to_1d_array(d2err_raw)

#         # Presence flag (some states may not have d2_err)
#         has_err = d2_err.size > 0 and np.any(np.isfinite(d2_err))
#         n_states_with_err += int(has_err)

#         # Harmonize lengths: truncate all to the common min
#         sizes = [centers.size, d2_vals.size] + ([d2_err.size] if has_err else [])
#         n_pts = min(sizes) if min(sizes) > 0 else 0
#         print(f"   State {i}: {state} - requested points={centers.size}, actual used={n_pts}")

#         if n_pts == 0:
#             print("     ⚠️ Empty state - no data points")
#             continue

#         centers = centers[:n_pts]
#         d2_vals = d2_vals[:n_pts]
#         d2_err  = (d2_err[:n_pts] if has_err else np.full(n_pts, np.nan, dtype=float))

#         # Ranges
#         try:
#             print(f"     Time range: {float(np.nanmin(centers))} to {float(np.nanmax(centers))}")
#         except ValueError:
#             print("     Time range: (all NaN)")

#         try:
#             print(f"     D2 range: {float(np.nanmin(d2_vals)):.4f} to {float(np.nanmax(d2_vals)):.4f}")
#         except ValueError:
#             print("     D2 range: (all NaN)")

#         if np.any(np.isfinite(d2_err)):
#             try:
#                 dmin, dmax = float(np.nanmin(d2_err)), float(np.nanmax(d2_err))
#                 print(f"     D2_ERR range: {dmin:.4f} to {dmax:.4f}")
#             except ValueError:
#                 print("     D2_ERR range: (all NaN)")
#         else:
#             print("     D2_ERR: missing → filled with NaN")

#         all_centers.append(centers)
#         all_d2.append(d2_vals)
#         all_d2err.append(d2_err)
#         all_states.append(np.array([state]*n_pts, dtype=object))

#     if len(all_centers) == 0:
#         raise ValueError("No data found in any sleep state - all states are empty")

#     # Concatenate
#     all_centers = np.concatenate(all_centers)
#     all_d2      = np.concatenate(all_d2)
#     all_d2err   = np.concatenate(all_d2err)
#     all_states  = np.concatenate(all_states)

#     # DataFrame (include d2_err)
#     df = pd.DataFrame({
#         'center_s': all_centers.astype(float, copy=False),
#         'd2':       all_d2.astype(float, copy=False),
#         'd2_err':   all_d2err.astype(float, copy=False),
#         'state':    all_states.astype(str)
#     }).sort_values('center_s').reset_index(drop=True)

#     print(f"\n✅ Complete timeline created:")
#     print(f"📊 Total data points: {len(df)}")
#     print(f"📊 Time range: {df['center_s'].min()} to {df['center_s'].max()}")
#     print(f"📊 Duration: {(df['center_s'].max() - df['center_s'].min()):.1f} seconds")
#     print(f"📊 State distribution:")
#     print(df['state'].value_counts())

#     # Empty-state tracker
#     empty_states = []
#     for ri in iter_results:
#         st = _to_state_str(ri.get('state','UNKNOWN')) if hasattr(ri,'get') else \
#              _to_state_str(ri['state']) if (isinstance(ri,dict) or hasattr(ri,'__getitem__')) and 'state' in ri else 'UNKNOWN'
#         csz = np.size(np.atleast_1d(ri.get('center', []))) if hasattr(ri,'get') else \
#               np.size(np.atleast_1d(ri['center'])) if (isinstance(ri,dict) or hasattr(ri,'__getitem__')) and 'center' in ri else 0
#         if csz == 0:
#             empty_states.append(st)

#     meta = {
#         "animal_name": animal_name,
#         "day": day,
#         "region": region,
#         "path": mat_path,
#         "params": params,
#         "columns": list(df.columns),
#         "shape": df.shape,
#         "n_states": len(iter_results),
#         "states_with_d2_err": n_states_with_err,
#         "empty_states": empty_states
#     }

#     return df, meta



# # Test the complete loading function
# try:
#     ca1_complete_df, ca1_complete_meta = load_d2_timeline_complete(basepath, "CA1")
    
#     # pfc_complete_df, pfc_complete_meta = load_d2_timeline_complete(basepath, "PFC")

    
# except Exception as e:
#     print(f"❌ Failed: {e}")
#     import traceback
#     traceback.print_exc()

# print("\n" + "="*60)

In [43]:
# import numpy as np
# import pandas as pd

# def _to_epochs_array(epochs):
#     """Return sorted float64 array of shape (n,2) or (0,2)."""
#     arr = getattr(epochs, "data", epochs)
#     arr = np.asarray(arr, dtype=float)
#     if arr.size == 0:
#         return np.empty((0, 2), dtype=float)
#     if arr.ndim != 2 or arr.shape[1] != 2:
#         raise ValueError("epochs must be (n,2) [start, stop]")
#     # keep only proper intervals
#     arr = arr[arr[:,1] > arr[:,0]]
#     if arr.size == 0:
#         return np.empty((0, 2), dtype=float)
#     # sort by start
#     idx = np.argsort(arr[:,0])
#     return arr[idx]

# def _count_overlaps_per_window(centers, half_w, starts, stops):
#     """
#     Count events that overlap each window [c-half_w, c+half_w].
#     Two-pointer sweep; O(n + m) once arrays are sorted.
#     """
#     n = len(centers)
#     counts = np.zeros(n, dtype=int)
#     i = 0  # index over events

#     # For convenience ensure arrays are numpy and sorted
#     centers = np.asarray(centers, dtype=float)
#     starts  = np.asarray(starts,  dtype=float)
#     stops   = np.asarray(stops,   dtype=float)

#     # advance i to first event that could possibly overlap the first window
#     first_start_needed = centers[0] - half_w
#     while i < len(starts) and stops[i] <= first_start_needed:
#         i += 1

#     j = i
#     for k, c in enumerate(centers):
#         w0 = c - half_w
#         w1 = c + half_w

#         # move i up until events stop strictly before window
#         while i < len(starts) and stops[i] <= w0:
#             i += 1
#         # advance j to include all events that start before window end
#         if j < i:
#             j = i
#         while j < len(starts) and starts[j] < w1:
#             j += 1

#         # events that possibly overlap are in [i, j)
#         # filter by actual overlap: start < w1 and stop > w0 (already enforced by i/j logic)
#         counts[k] = max(0, j - i)

#     return counts

# def sliding_event_rate(df, event_epochs, window_size_s, center_col="center_s", rate_col_name="rate_hz"):
#     """
#     Compute event rate per D2 window.
#       - df: DataFrame with a column of centers (seconds)
#       - event_epochs: (n,2) [start, stop] in seconds (e.g., ripple_epochs.data)
#       - window_size_s: float (e.g., 30.0)
#       - returns a numpy array of rates aligned to df rows
#     """
#     ep = _to_epochs_array(event_epochs)
#     if ep.shape[0] == 0 or len(df) == 0:
#         return np.zeros(len(df), dtype=float)

#     half_w = float(window_size_s) / 2.0
#     starts, stops = ep[:,0], ep[:,1]

#     # Safety: ensure starts/stops are sorted together
#     order = np.argsort(starts)
#     starts = starts[order]
#     stops  = stops[order]

#     centers = df[center_col].to_numpy(dtype=float)
#     counts = _count_overlaps_per_window(centers, half_w, starts, stops)

#     # rate in Hz (events per second)
#     return counts / float(window_size_s)

# # ---------- Usage ----------
# # Assumptions:
# # - ca1_complete_df is the D2 timeline DataFrame with columns: ['center_s','d2','state', ...]
# # - ripple_epochs.data and barrage_epochs.data are (n,2) arrays in seconds
# # - window_size = 30.0  (seconds)
# # - step_size   = 2.0   (seconds)  # not needed explicitly for the calculation

# window_size = 30.0

# # Compute ripple and barrage rates aligned to each D2 center
# ca1_complete_df["ripple_rate_hz"]  = sliding_event_rate(
#     ca1_complete_df, ripple_epochs.data, window_size, center_col="center_s", rate_col_name="ripple_rate_hz"
# )
# ca1_complete_df["barrage_rate_hz"] = sliding_event_rate(
#     ca1_complete_df, barrage_epochs.data, window_size, center_col="center_s", rate_col_name="barrage_rate_hz"
# )

# print(ca1_complete_df[["center_s","d2","ripple_rate_hz","barrage_rate_hz"]].head())

In [44]:
# from pathlib import Path
# import pandas as pd

# # Inputs:
# # ca1_complete_df : pandas DataFrame
# # animal_name     : str
# # day             : int | str
# # brain_region    : str

# out_dir = Path(r"D:\Documents\GitHub\Sleep_criticaltiy\data\ripple_barr\d2_uncertainty")
# out_dir.mkdir(parents=True, exist_ok=True)

# fname = f"{animal_name}_{day}_{brain_region}.pkl"
# save_path = out_dir / fname

# ca1_complete_df.to_pickle(save_path)
# print(f"Saved to: {save_path}")

# # Later, to load:
# # df = pd.read_pickle(save_path)

cross state d2

In [50]:
# # Save all data to pickle file
# import pickle
# import os

# # Create save directory if it doesn't exist
# save_path = r"D:\Documents\GitHub\Sleep_criticaltiy\data\cross_state\mice_d2_barr_ripple"
# os.makedirs(save_path, exist_ok=True)

# # Extract animal name and day from metadata
# animal_name = ca1_complete_meta.get('animal_name', 'unknown')
# day = ca1_complete_meta.get('day', 'unknown')

# # Create filename
# filename = f"{animal_name}_{day}.pkl"
# full_path = os.path.join(save_path, filename)

# # Prepare data dictionary to save
# data_to_save = {
#     # 'box_df_pfc': box_df_pfc,
#     # 'meta_pfc': meta_pfc,
#     'ca1_complete_meta': ca1_complete_meta,
#     # 'pfc_complete_meta': pfc_complete_meta,
#     'ca1_complete_df': ca1_complete_df,  # Also save the complete dataframes
#     # 'pfc_complete_df': pfc_complete_df,
#     'save_info': {
#         'created_date': pd.Timestamp.now().isoformat(),
#         'notebook': 'MUA_sleepstage.ipynb',
#         'description': 'd2 and ripple/barrage rates, box plots by state, correlations',
#     }
# }

# # # Save to pickle file
# try:
#     with open(full_path, 'wb') as f:
#         pickle.dump(data_to_save, f, protocol=pickle.HIGHEST_PROTOCOL)
    
#     print(f"✅ Successfully saved data to:")
#     print(f"   📁 {full_path}")
#     print(f"   📊 File size: {os.path.getsize(full_path) / 1024:.1f} KB")
    
#     # Print summary of saved data
#     print(f"\n📊 Saved data summary:")
#     for key, value in data_to_save.items():
#         if isinstance(value, pd.DataFrame):
#             print(f"   {key}: DataFrame ({value.shape[0]} rows, {value.shape[1]} cols)")
#         elif isinstance(value, dict):
#             print(f"   {key}: Dict ({len(value)} keys)")
#         else:
#             print(f"   {key}: {type(value).__name__}")
    
#     # Verify the file was saved correctly
#     print(f"\n🔍 Verification:")
#     if os.path.exists(full_path):
#         print(f"   ✅ File exists and is readable")
        
#         # Quick test load
#         with open(full_path, 'rb') as f:
#             test_data = pickle.load(f)
#         print(f"   ✅ File loads successfully")
#         print(f"   📊 Contains {len(test_data)} data objects")
        
#         # Check data integrity
#         if 'box_df_ca1' in test_data and len(test_data['box_df_ca1']) > 0:
#             print(f"   ✅ CA1 box data: {len(test_data['box_df_ca1'])} rows")
#         if 'box_df_pfc' in test_data and len(test_data['box_df_pfc']) > 0:
#             print(f"   ✅ PFC box data: {len(test_data['box_df_pfc'])} rows")
            
#     else:
#         print(f"   ❌ File was not created properly")
        
# except Exception as e:
#     print(f"❌ Error saving data: {e}")
#     import traceback
#     traceback.print_exc()


# print(f"\n" + "="*60)
# print(f"DATA SAVE SUMMARY")
# print(f"="*60)
# print(f"Animal: {animal_name}")
# print(f"Day: {day}")
# print(f"Main file: {filename}")
# print(f"Directory: {save_path}")
# print(f"Total objects saved: {len(data_to_save)}")
# print(f"="*60)

novel and familiar memory

In [ ]:
# simplest: replace the substring anywhere in the name
# epoch_df["name"] = epoch_df["name"].str.replace("midseelp", "midsleep", regex=False)

In [ ]:
# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt

# import re

# def label_epoch_from_name(name: str) -> str:
#     s = (str(name) or "").lower()

#     if "presleep" in s:  return "presleep"
#     if "midsleep"  in s: return "midsleep"
#     if "postsleep" in s: return "postsleep"

#     # --- task / T-maze markers ---
#     # match 't_' or 't1_' / 't2_' / ... anywhere (word-like), plus common aliases
#     if (re.search(r'(?:^|[^a-z])t(?:_|[0-9])', s)          # t_ or t#_
#         or "tmaze" in s or "t-maze" in s
#         or "freet" in s or "force" in s
#         or "task" in s):
#         return "task"

#     return "other"

# # --- normalize possible Data Wrangler column names ---
# def _normalize_epoch_columns(df: pd.DataFrame) -> pd.DataFrame:
#     tmp = df.copy()
#     # strip spaces and leading '#' from column names
#     tmp.columns = [c.strip().lstrip("#").strip() for c in tmp.columns]
#     # make sure required columns exist (startTime/stopTime/name)
#     rename_map = {}
#     for c in tmp.columns:
#         cl = c.lower()
#         if cl == "starttime": rename_map[c] = "startTime"
#         if cl == "stoptime":  rename_map[c] = "stopTime"
#         if cl == "name":      rename_map[c] = "name"
#     if rename_map:
#         tmp = tmp.rename(columns=rename_map)
#     missing = [c for c in ("name", "startTime", "stopTime") if c not in tmp.columns]
#     if missing:
#         raise KeyError(f"epoch_df missing required columns: {missing}")
#     return tmp

# def annotate_epochs(ca_df: pd.DataFrame, epoch_df: pd.DataFrame,
#                     enumerate_labels=("postsleeppp",),  # labels to number by order
#                     max_tasks=3):
#     tmp = _normalize_epoch_columns(epoch_df)[["name","startTime","stopTime"]].copy()
#     tmp["epoch_group"] = [label_epoch_from_name(n) for n in tmp["name"]]
#     tmp = tmp.dropna(subset=["startTime","stopTime"])
#     tmp = tmp[tmp["startTime"] < tmp["stopTime"]].copy()
#     tmp = tmp.sort_values("startTime").reset_index(drop=True)

#     # number selected labels (e.g., midsleep1, midsleep2, ...)
#     for lab in enumerate_labels:
#         m = tmp["epoch_group"].eq(lab)
#         if m.any():
#             idxs = tmp.index[m].tolist()
#             for k, i in enumerate(idxs, start=1):
#                 tmp.loc[i, "epoch_group"] = f"{lab}{k}"

#     # number first few task blocks as before
#     m_task = tmp["epoch_group"].eq("task")
#     if m_task.any():
#         task_order = tmp.index[m_task].tolist()
#         for i, idx in enumerate(task_order[:max_tasks], start=1):
#             tmp.loc[idx, "epoch_group"] = f"task{i}"

#     intervals = pd.IntervalIndex.from_arrays(tmp["startTime"].to_numpy(),
#                                              tmp["stopTime"].to_numpy(),
#                                              closed="left")
#     tmp = tmp.set_index(intervals)

#     ca = ca_df.copy()
#     centers = ca["center_s"].to_numpy()
#     idx = tmp.index.get_indexer(centers)
#     ca["epoch_group"] = np.where(idx >= 0, tmp.iloc[idx]["epoch_group"].to_numpy(), np.nan)
#     return ca


# # -------- 1) annotate ca1_complete_df with epoch_group --------
# ca_annot = annotate_epochs(ca1_complete_df, epoch_df)

# # -------- 2) filter to groups & make boxplots --------
# wanted_groups = ["presleep", "midsleep", "postsleep"]  # add "task" for a 4th box

# def plot_box_for_state(df_annot, state_label, groups=wanted_groups, title=None):
#     d = (df_annot
#          .loc[(df_annot["state"].astype(str).str.upper()==state_label.upper())
#               & (df_annot["epoch_group"].isin(groups))]
#          .copy())
#     if d.empty:
#         print(f"No rows for state={state_label}")
#         return

#     # order & tidy
#     d["epoch_group"] = pd.Categorical(d["epoch_group"], categories=groups, ordered=True)

#     # --- boxplot (matplotlib) ---
#     fig, ax = plt.subplots(figsize=(5.2,4.2))
#     data = [d.loc[d["epoch_group"]==g, "d2"].dropna().values for g in groups]
#     ax.boxplot(data, labels=[g.replace("sleep"," sleep") for g in groups],
#                showfliers=False, widths=0.6)
#     ax.set_xlabel("Epoch")
#     ax.set_ylabel("D²")
#     ax.set_title(title or f"{state_label}: D² by epoch")
#     for s in ("top","right"): ax.spines[s].set_visible(False)
#     plt.tight_layout()
#     plt.show()

#     # optional: quick stats readout (Kruskal–Wallis + pairwise Wilcoxon, paired only if the same file/session needed)
#     try:
#         from scipy.stats import kruskal, mannwhitneyu
#         stats_groups = [g for g in data if len(g) >= 5]
#         if len(stats_groups) >= 2:
#             H, p = kruskal(*stats_groups)
#             print(f"[{state_label}] Kruskal–Wallis across groups: H={H:.2f}, p={p:.3e}")
#         # pairwise (unpaired) Mann–Whitney as a lightweight summary
#         for i in range(len(groups)):
#             for j in range(i+1, len(groups)):
#                 gi, gj = data[i], data[j]
#                 if len(gi)>=5 and len(gj)>=5:
#                     U, p = mannwhitneyu(gi, gj, alternative="two-sided")
#                     print(f"  {groups[i]} vs {groups[j]}: U={U:.0f}, p={p:.3e}")
#     except Exception as e:
#         print("Stats skipped:", e)

# # NREM: compare pre/mid/post sleep
# ca_annot = annotate_epochs(ca1_complete_df, epoch_df)
# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt
# import seaborn as sns

# # ---------- build normalized df ----------
# def first_n_per_epoch(df, n=1200):
#     """Return first n rows per epoch_group after sorting by center_s."""
#     return (df.sort_values("center_s")
#               .groupby("epoch_group", group_keys=False, observed=True)
#               .head(n))

# sleep_groups = ["presleep", "midsleep", "postsleep"]

# # 1) NREM only, sleep epochs only
# d = ca_annot.copy()
# d["state"] = d["state"].astype(str).str.upper()
# d = d[(d["state"] == "NREM") & (d["epoch_group"].isin(sleep_groups))].copy()

# # 2) take first 1200 rows per epoch
# d = first_n_per_epoch(d, n=1200)

# # 3) presleep mean
# pre = d[d["epoch_group"] == "presleep"]["d2"].dropna().to_numpy()
# if pre.size == 0:
#     raise ValueError("No presleep NREM rows found (first 1200).")
# m_pre = float(np.nanmean(pre))

# # safety for zero (very unlikely, but guard anyway)
# eps = 1e-12 if m_pre == 0 else 0.0

# # 4) normalize midsleep & postsleep
# d2_rel = d[d["epoch_group"].isin(["midsleep", "postsleep"])].copy()
# d2_rel["d2_rel"] = (d2_rel["d2"] - m_pre) / (d2_rel["d2"] + m_pre + eps)
# d2_rel["epoch_group"] = pd.Categorical(
#     d2_rel["epoch_group"], categories=["midsleep", "postsleep"], ordered=True
# )

# # ---------- plot (only midsleep & postsleep) ----------
# plt.figure(figsize=(5.4, 4.0))
# sns.boxplot(data=d2_rel, x="epoch_group", y="d2_rel", showfliers=False, width=0.6)
# sns.stripplot(data=d2_rel, x="epoch_group", y="d2_rel",
#               color="k", alpha=0.25, size=2, linewidth=0, dodge=False)
# plt.axhline(0, lw=1, ls="--", color="gray")  # zero = equal to presleep mean
# plt.xlabel("")
# plt.ylabel(r"(sleep − presleep) / (sleep + presleep)")
# plt.title("NREM (first 1200 rows): midsleep & postsleep vs presleep baseline")
# ax = plt.gca()
# ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
# plt.tight_layout(); plt.show()

# # (optional) quick text readout
# print(f"Presleep mean D² (NREM, first 1200 rows): {m_pre:.4g}")
# for g in ["midsleep", "postsleep"]:
#     vals = d2_rel.loc[d2_rel["epoch_group"]==g, "d2_rel"].dropna().to_numpy()
#     if vals.size:
#         print(f"{g}: n={vals.size}, median={np.median(vals):.3f}, mean={np.mean(vals):.3f}")

In [ ]:
# from pathlib import Path
# out_dir=r"D:\Documents\GitHub\Sleep_criticaltiy\data\novel_familiar\novel"
# # --- save d2_rel ---
# Path(out_dir).mkdir(parents=True, exist_ok=True)
# save_path = Path(out_dir) / f"{animal_name}_{day}.pkl"
# d2_rel.to_pickle(save_path)
# print(f"Saved: {save_path}")
